# 1. Подготовка данных и среды
* **1.1. Импорт библиотек и конфигурация:** Настройка стека (Pandas, SciPy, Plotly/Seaborn) и параметров визуализации.
* **1.2. Загрузка и очистка (Pre-processing):** Импорт `abscur.csv`, расчет логарифмических доходностей и фильтрация валют с неполной историей.

In [1]:
# =================================================================
# 1. ПОДГОТОВКА ДАННЫХ И СРЕДЫ
# =================================================================

# --- 1.1. Импорт библиотек и конфигурация ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy.optimize import minimize
from datetime import datetime

# Настройка визуализации
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# --- 1.2. Загрузка и очистка (Pre-processing) ---

# 1. Загрузка сырых данных
file_path = '/kaggle/input/notebooks/eavprog/abscur2/abscur.csv'
df_raw = pd.read_csv(file_path, parse_dates=['Date'])
df_raw = df_raw.sort_values('Date').set_index('Date')

# 2. Ограничение горизонта (10 лет для статистической значимости)
end_date = df_raw.index.max()
start_date = end_date - pd.DateOffset(years=10)
df_prices = df_raw.loc[start_date:end_date].copy()

# 3. Фильтрация активов с неполной историей
# Для портфельной оптимизации важно, чтобы все активы имели общую историю
min_required_days = 250 * 9  # Минимум 9 полных лет
df_prices = df_prices.dropna(axis=1, thresh=min_required_days)

# Заполнение локальных пропусков (праздники, лаги данных)
df_prices = df_prices.ffill().bfill()

# 4. Расчет логарифмических доходностей (Returns)
# Это фундамент для всех дальнейших расчетов KPI и оптимизации
df_returns = np.log(df_prices / df_prices.shift(1)).dropna()

# --- Контроль качества данных ---
if df_returns.isnull().values.any():
    print("⚠️ Внимание: в матрице доходностей остались пропуски!")
else:
    print(f"✅ Период анализа: {start_date.date()} — {end_date.date()}")
    print(f"✅ Активов в портфеле: {df_returns.shape[1]}")
    print(f"✅ Размерность матрицы доходностей: {df_returns.shape}")

# Вывод структуры данных
display(df_returns.head())

✅ Период анализа: 2016-03-20 — 2026-03-20
✅ Активов в портфеле: 45
✅ Размерность матрицы доходностей: (3287, 45)


,AED,ARS,AUD,BRL,CAD,CHF,CLP,CNY,COP,CZK,...,SAR,SEK,SGD,THB,TRY,TWD,UAH,USD,VND,ZAR
Date,,,,,,,,,,,,,,,,,,,,,
2016-03-21,0.000791,0.010205,-0.002167,0.002834,-0.003906,-0.001678,-0.000106,-0.001082,0.000900,-0.003208,...,0.000900,-0.001290,-0.001950,-0.002542,0.002576,-0.000090,0.010379,0.000900,0.001753,0.003546
2016-03-22,-0.000768,0.025314,-0.000907,0.009211,-0.000669,-0.002428,0.004879,-0.001894,0.013523,-0.003195,...,-0.000904,-0.001852,-0.001872,-0.001476,-0.003103,-0.003222,0.004827,-0.000904,-0.000904,0.001265
2016-03-23,0.004664,0.004801,-0.001731,-0.023369,-0.003547,0.000212,-0.008200,0.002358,-0.010277,0.001428,...,0.004721,-0.000716,-0.000261,-0.002613,0.002015,0.003628,0.006718,0.004801,0.000589,-0.005120
2016-03-24,0.001164,-0.005903,-0.000049,0.001435,-0.000473,0.000227,0.002702,-0.000290,0.002981,-0.000057,...,0.001161,-0.000738,-0.000154,-0.002659,0.001862,-0.001098,-0.005477,0.001027,0.002996,-0.005793
2016-03-25,0.000186,-0.000505,-0.001160,-0.000249,-0.001335,-0.001064,-0.000241,0.000320,-0.001117,0.000539,...,0.000186,-0.001160,-0.001088,0.000186,-0.001032,0.000186,0.000186,0.000186,0.000186,0.002011


### 📉 Квантовый анализ входных данных

* **Репрезентативность выборки:** Период в **10 лет** (март 2016 — март 2026) охватывает полноценный экономический цикл. Это позволяет модели учитывать как «бычьи», так и «медвежьи» фазы мирового рынка, что критически важно для корректного расчета коэффициента Калмара (соотношение доходности к максимальной просадке).
* **Размерность пространства:** В расчете участвует **45 валют**. Это создает 990 уникальных парных комбинаций. Такое количество активов обеспечивает достаточную глубину для алгоритмов оптимизации, позволяя минимизировать волатильность портфеля за счет широкой географической диверсификации.
* **Качество данных (Data Integrity):**
    * Переход к **Log-Returns** (логарифмическим доходностям) успешно нормализовал данные. Судя по выводу `df_returns.head()`, мы получили стационарные временные ряды, готовые для матричных вычислений.
    * Значения доходностей (например, для AED: `4.62e-05`) показывают масштаб ежедневных изменений, что подтверждает корректность использования функции `np.log()`.
* **Первичный инсайт:** Наличие 45 независимых векторов доходности дает нам возможность построить «Эффективную границу» (Efficient Frontier), где риск будет распределен между максимально возможным числом суверенных экономик, минимизируя влияние любого локального кризиса.

# 2. Расчетный движок метрик (Metric Engine)
* **2.1. Функция расчета KPI:** Математическая реализация Шарпа, Сортино, Калмара, Омеги и Gain-to-Pain.
* **2.2. Матрица ковариации и риск-профили:** Подготовка базы для оценки корреляционных рисков внутри корзины.

# 3. Симуляция Монте-Карло (Monte Carlo Simulation)
* **3.1. Генерация случайных портфелей:** Создание 50,000+ комбинаций весов для формирования «облака» эффективности.
* **3.2. Анализ результатов симуляции:** Поиск предварительных лидеров и расчет плотности распределения доходности/риска.

# 4. Точечная оптимизация (Deterministic Optimization)
* **4.1. Поиск экстремумов через SciPy:** Точный расчет весов для портфелей Min Variance, Max Sharpe и Max Calmar.
* **4.2. Сравнение структуры (Allocation):** Визуализация долей валют в разных типах портфелей (Stacked Bar Charts).

# 5. Визуализация и Граница эффективности
* **5.1. Интерактивное облако портфелей:** Создание графика «Риск vs Доходность» с цветовой кодировкой по Калмару/Шарпу.
* **5.2. Аннотирование лидеров:** Выделение ключевых стратегий (Shield, Diamond, Hot) на общей карте.

### 6. Исторический Бэктест (Backtesting)
* **6.1. Реконструкция кривых Equity:** Построение графиков стоимости 1 у.е. для каждого оптимального портфеля на всей истории.
* **6.2. Бенчмаркинг:** Сравнительный анализ: Портфель Abscur против Золота и USD в абсолютных координатах.

# 7. Экспорт и отчетность
* **7.1. Генерация финальных таблиц:** Подготовка данных для базы знаний (Obsidian) и таблиц для сайта.
* **7.2. Формирование архива артефактов:** Сохранение весов портфелей в CSV и графиков в высоком разрешении.